# Splitting data

In [1]:
import seaborn as sns
import pandas as pd 

df = sns.load_dataset("tips")
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [2]:
df.shape

(244, 7)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [4]:
X = df.drop(columns="total_bill")
# X = df.drop("total_bill", axis=1)
y = df["total_bill"]

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

In [6]:
print("Total Samples:", len(df))
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Total Samples: 244
Training Samples: 195
Testing Samples: 49


# Feature scaling
### Why Feature Scaling?

Feature scaling is required because **machine learning models are sensitive to the scale of input features**. When features have different ranges (e.g., `total_bill` in tens vs. `size` in small integers), larger-magnitude features can **dominate distance calculations, gradient updates, or regularization**, leading to biased or unstable learning. Scaling ensures that **each feature contributes proportionally**, improves **convergence speed**, and is **essential** for distance-based and gradient-based algorithms (e.g., KNN, SVM, Linear/Logistic Regression, Neural Networks).


---
For the **`tips` dataset**, feature scaling applies **only to numerical features**.

### Numerical features (scaling required)

* `total_bill` **(target – do NOT scale in preprocessing)**
* `tip`
* `size`

➡ **Scale only**:

* `tip`
* `size`

### Categorical features (NO scaling)

* `sex`
* `smoker`
* `day`
* `time`

### Summary (one line)

Scale **`tip` and `size`**; encode **categorical columns** later; **never scale the target** (`total_bill`) in preprocessing.


---
Because **feature scaling is meaningful only for numerical features**, not categorical ones.

### Explanation (concise and precise)

* **Numerical features (`tip`, `size`)** represent **quantitative magnitudes**.
  Their values participate in arithmetic operations (distance, gradients, dot products), so unequal scales can **bias the model**.

* **Categorical features (`sex`, `smoker`, `day`, `time`)** represent **labels, not magnitudes**.
  Scaling them would imply a false numeric relationship (e.g., `day=3` being “greater” than `day=1`), which is **mathematically incorrect**.

* **Target variable (`total_bill`)** is **not scaled during preprocessing** because:

  * It is not an input feature
  * Scaling it would distort error interpretation and evaluation metrics

### Bottom line (interview-ready)

Scale **only numerical input features** because models learn from magnitudes; categorical data must be **encoded**, not scaled, and the target is handled separately.


In [7]:
# numerical columns to scale
num_cols= ["tip", "size"]
num_cols

['tip', 'size']

In [8]:
from sklearn.preprocessing import StandardScaler


$$
\frac{x - x_{min}}{x_{max} - x_{min}}
$$

This formula belongs to **MinMaxScaler**, not StandardScaler.

---

 **StandardScaler** actually does

$$
z = \frac{x - \mu}{\sigma}
$$

Where:

* $\mu$ = mean of the feature (from **training data**)
* $\sigma$ = standard deviation (from **training data**)

### Resulting scale

* Mean → **0**
* Standard deviation → **1**
* Values are **not bounded** (can be negative or > 1)



---

### Feature Scaling — Small Cheat Sheet

Use this as a **decision table**.

---

### **StandardScaler**

**When to use**

* Features are roughly **normally distributed**
* Using **Linear / Logistic Regression**
* **SVM**, **Neural Networks**
* Algorithms using **gradients or dot products**

**Why**

* Centers data at 0
* Unit variance
* Faster convergence

---

### **MinMaxScaler**

**When to use**

* You need **fixed bounds** (0–1)
* **Neural networks**, especially with sigmoid / tanh
* Image-like or normalized inputs

**Why**

* Preserves relative distances
* Keeps values in a strict range

---

### **RobustScaler**

**When to use**

* Data contains **outliers**
* Skewed distributions

**Why**

* Uses **median and IQR**
* Outlier-resistant

---

### **MaxAbsScaler**

**When to use**

* Sparse data (e.g., text vectors)
* Want to preserve sparsity

**Why**

* Scales by max absolute value
* No centering

---

### **One-line rule (interview ready)**

> Use **StandardScaler by default**, **MinMaxScaler for bounded inputs**, and **RobustScaler when outliers dominate**.

---

# Standard Scaler

In [9]:
scaler = StandardScaler()


In [10]:
X_train_scaled = X_train.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])

* **`X_train.copy()`** → prevents modifying the original training data in-place.
* **`scaler.fit_transform()`** → fits (learns mean/std) **and** transforms in one step; `fit()` alone only learns parameters and does not scale the data.


---
Use `fit_transform()` on training data to learn and apply scaling, and `transform()` on test data so the model sees test data scaled using **only training statistics**, preventing data leakage and ensuring valid evaluation.


In [11]:
# Transform test data using training statistics
X_test_scaled = X_test.copy()
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])
# Applies training-learned scaling parameters to test data without leaking information

In [12]:
X_train_scaled[num_cols].mean().round(10), X_train_scaled[num_cols].std().round(10)

(tip    -0.0
 size   -0.0
 dtype: float64,
 tip     1.002574
 size    1.002574
 dtype: float64)


> The means appear as `-0.0` due to floating-point precision and rounding; they are effectively zero, confirming correct centering. The standard deviation is ~1 (not exactly 1) because `StandardScaler` uses population std (`ddof=0`), while pandas’ `.std()` uses sample std (`ddof=1`).


---
### `ddof` — Degrees of Freedom (simple explanation)

`ddof` controls **how the standard deviation is calculated**.

---

### **`ddof = 0` (Population standard deviation)**

$$
\sigma = \sqrt{\frac{1}{N} \sum (x - \mu)^2}
$$

* Divides by **N**
* Assumes data is the **entire population**
* Used by **StandardScaler**
* Default in **NumPy**

---

### **`ddof = 1` (Sample standard deviation)**

$$
s = \sqrt{\frac{1}{N-1} \sum (x - \bar{x})^2}
$$

* Divides by **N − 1**
* Corrects bias when estimating population from a sample
* Default in **pandas `.std()`**
* Common in **statistics**

---

### Why this matters here

* `StandardScaler` → `ddof = 0`
* `pandas std()` → `ddof = 1`
* Hence you saw **1.002574 instead of exactly 1**

---

### One-line takeaway

> `ddof=0` is for population scaling (ML preprocessing), `ddof=1` is for sample estimation (statistics).


# Min Max scaler(Not applied in the tips, just for reference) 

In [13]:
from sklearn.preprocessing import MinMaxScaler

In [14]:
minmax_scaler = MinMaxScaler()


In [15]:
# Fit on training data and transform training data
X_train_minmax = X_train.copy()
X_train_minmax[num_cols] = minmax_scaler.fit_transform(X_train[num_cols])


In [16]:
# Transform test data using training min/max
X_test_minmax = X_test.copy()
X_test_minmax[num_cols] = minmax_scaler.transform(X_test[num_cols])


In [17]:
X_train_minmax[num_cols].min(), X_train_minmax[num_cols].max()


(tip     0.0
 size    0.0
 dtype: float64,
 tip     1.0
 size    1.0
 dtype: float64)

After MinMax scaling, training numerical features have minimum 0 and maximum 1, confirming correct rescaling using training data statistics.

--- 
For this project and for syllabus clarity, the correct approach is as follows: in real-world ML practice for the `tips` dataset, **One-Hot Encoding** is used, while for syllabus and conceptual understanding, **both Label Encoding and One-Hot Encoding** should be studied. The categorical columns in this dataset (`sex`, `smoker`, `day`, `time`) are **nominal**, meaning they have no inherent order. Using Label Encoding here would incorrectly impose an ordinal relationship (for example, encoding days as 0, 1, 2, 3 may make the model assume `3 > 1`, which is meaningless). Therefore, One-Hot Encoding is the correct and safe choice for this dataset. Label Encoding is not wrong, but it is **situational** and is mainly appropriate for encoding classification targets, ordinal categories (such as low < medium < high), or in some tree-based models. In summary, **Label Encoding is covered for syllabus completeness**, while **One-Hot Encoding is the method actually applied in this project**, and next we will first briefly demonstrate Label Encoding for concept clarity and then apply One-Hot Encoding properly using pipelines and a ColumnTransformer.


In [18]:
# just to cover topics
from sklearn.preprocessing import LabelEncoder

education = ["primary", "secondary", "graduate", "secondary", "primary"]

le = LabelEncoder()
encoded_education = le.fit_transform(education)
encoded_education, le.classes_

(array([1, 2, 0, 2, 1]),
 array(['graduate', 'primary', 'secondary'], dtype='<U9'))

In [19]:
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

sample_tips = sns.load_dataset("tips")

le = LabelEncoder()
sample_tips['sex_encoded'] = le.fit_transform(sample_tips["sex"])  # Adds new encoded column $$code:1$$
print(sample_tips.head())
print(le.classes_)

   total_bill   tip     sex smoker  day    time  size  sex_encoded
0       16.99  1.01  Female     No  Sun  Dinner     2            0
1       10.34  1.66    Male     No  Sun  Dinner     3            1
2       21.01  3.50    Male     No  Sun  Dinner     3            1
3       23.68  3.31    Male     No  Sun  Dinner     2            1
4       24.59  3.61  Female     No  Sun  Dinner     4            0
['Female' 'Male']


Continue with Dataset

In [20]:
from sklearn.preprocessing import OneHotEncoder

Use pd.get_dummies() for exploration; use OneHotEncoder for training, evaluation, and production pipelines.

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [22]:
# Columns to one hot encode
cat_cols = ["sex", "smoker","day","time"]
cat_cols

['sex', 'smoker', 'day', 'time']

In [23]:
ohe = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
X_train_ohe = ohe.fit_transform(X_train[cat_cols])

handle_unknown="ignore" → unseen categories in test data won’t crash the model

sparse_output=False → returns a NumPy array (easier to inspect now)

---
`handle_unknown="ignore"` is used to make models robust to unseen categories. For example, if the training data contains only `day = $$"Sun", "Sat"$$` and the test or production data later includes a new value like `"Fri"`, the default behavior (`handle_unknown="error"`) would cause the pipeline to fail during transformation. When `handle_unknown="ignore"` is enabled, the encoder does not crash; instead, it encodes the unseen category as an all-zero vector for that feature while keeping the pipeline running. This is critical in real projects because new categories inevitably appear in production, and models must handle them gracefully without immediate retraining.


---
When `sparse_output=True`, `OneHotEncoder` returns a **sparse matrix**, meaning only the positions of `1`s are stored while zeros are omitted, which significantly reduces memory usage for datasets with many categories. This setting is preferred for **large datasets, high-cardinality categorical features, text-like or wide feature spaces, and production pipelines**. In contrast, `sparse_output=False` returns a dense array, which is more suitable for **small datasets, learning, debugging, and easy conversion to pandas DataFrames**. In practice, use dense output for understanding and inspection, and sparse output for scalable, production-ready workflows.


In [24]:
# Transform test data using training-learned categories
X_test_ohe = ohe.transform(X_test[cat_cols])


### Below is just for refference

This manual combination is only for learning.
In real projects, this approach is error-prone and hard to maintain.
Pipeline + ColumnTransformer is the correct and production-ready solution.

In [25]:
# Get column names created by OneHotEncoder
ohe_feature_names = ohe.get_feature_names_out(cat_cols)

# Convert to DataFrame
X_train_ohe_df = pd.DataFrame(
    X_train_ohe,
    columns=ohe_feature_names,
    index=X_train.index
)

X_test_ohe_df = pd.DataFrame(
    X_test_ohe,
    columns=ohe_feature_names,
    index=X_test.index
)
X_train_ohe_df

,sex_Female,sex_Male,smoker_No,smoker_Yes,day_Fri,day_Sat,day_Sun,day_Thur,time_Dinner,time_Lunch
228,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
208,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
96,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
167,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
84,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
106,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
14,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
92,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
179,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [26]:
# Combine numerical and categorical features
X_train_final = pd.concat(
    [X_train_scaled[num_cols], X_train_ohe_df],
    axis=1
)

X_test_final = pd.concat(
    [X_test_scaled[num_cols], X_test_ohe_df],
    axis=1
)
X_train_final

,tip,size,sex_Female,sex_Male,smoker_No,smoker_Yes,day_Fri,day_Sat,day_Sun,day_Thur,time_Dinner,time_Lunch
228,-0.258033,-0.612141,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
208,-0.742114,-0.612141,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
96,0.639973,-0.612141,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
167,0.990757,1.519421,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
84,-0.742114,-0.612141,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.682067,-0.612141,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
14,-0.047563,-0.612141,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
92,-1.464729,-0.612141,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
179,0.324268,-0.612141,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


### Continue with pipeline

# Pipeline and ColumnTransformer

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

`Pipeline` and `ColumnTransformer` are scikit-learn tools that enable **clean, safe, and production-ready preprocessing**. A `Pipeline` chains multiple steps (such as scaling, encoding, and modeling) into a single object so that all operations are applied in the correct order using the same training-only logic during both training and testing. A `ColumnTransformer` works alongside a pipeline to apply **different preprocessing steps to different subsets of columns** (for example, scaling numerical features and one-hot encoding categorical features) while keeping everything synchronized. Together, they prevent data leakage, ensure consistent transformations, simplify experimentation, and make models easier to deploy and maintain.


In [28]:
# Numerical preprocessing pipeline
num_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])
# Categorical preprocessing pipeline
cat_pipeline = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])


- "scaler" → string name of the step

- StandardScaler() → actual transformer object

- "encoder" → name of the step

- OneHotEncoder() → actual encoder object

They are not variables and not special keywords.
They are labels used internally by the pipeline.


In a Pipeline, each step is a (name, object) pair stored in a list to preserve execution order and allow step referencing.

In [29]:
# Combine numerical and categorical pipelines
# Combine preprocessing for different column types
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ]
)


- "num" → name of the transformer block

- num_pipeline → what to apply

- num_cols → where to apply it

- "cat" → categorical block (same logic)

In [30]:
# Fit the preprocessor on training data and transform train/test

# Fit on training data and transform training data
X_train_prepared = preprocessor.fit_transform(X_train)

# Transform test data using the same fitted preprocessor
X_test_prepared = preprocessor.transform(X_test)


# Now using the pipeline 

In [31]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline


In [32]:
# Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])


In [33]:
# Train the model
model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [34]:
# Predict on test data
y_pred = model.predict(X_test)

y_pred[:5]


array([18.96162893, 14.50984202, 20.76231864, 33.76081431, 15.19014663])

In [35]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [36]:
# Regression metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

mae, mse, rmse, r2


(4.470042009997702, 31.873635550729933, 31.873635550729933, 0.6240808714290967)

# Finished

# Handling missing value(not in project pipeline, to finish topics)

In [37]:
df2 = sns.load_dataset("titanic")

In [38]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [39]:
df2.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


1. Drop deck

- Too many missing values (≈ 77\%)

- Imputation would add noise

2. Impute numerical columns

- age → SimpleImputer(strategy="mean")

3. Impute categorical columns

- embarked, embark_town → SimpleImputer(strategy="most_frequent")

4. All imputation will be inside a Pipeline

- Fit on train only

- No leakage

In [40]:
from sklearn.model_selection import train_test_split

df2= df2.drop(columns=["deck"])

X=df2.drop(columns="survived")
y=df2["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [41]:
# Identify numerical and categorical columns
num_cols2 = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols2 = X_train.select_dtypes(include=["object", "bool", "category"]).columns

print(num_cols2)
print(cat_cols2)


Index(['pclass', 'age', 'sibsp', 'parch', 'fare'], dtype='object')
Index(['sex', 'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive',
       'alone'],
      dtype='object')


In [42]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [43]:
# Numerical imputation pipeline
num_imputer2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean"))
])  

# Categorical imputation pipeline
cat_imputer2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])


In [44]:
from sklearn.compose import ColumnTransformer

In [45]:
# Combine numerical and categorical imputers
imputer_preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_imputer2, num_cols2),
        ("cat", cat_imputer2, cat_cols2)
    ]
)


In [46]:
# Fit on training data and transform training data
X_train_imputed = imputer_preprocessor.fit_transform(X_train)

# Transform test data using the same fitted imputer
X_test_imputed = imputer_preprocessor.transform(X_test)


In [47]:
# Convert to DataFrame temporarily for inspection
X_train_imputed_df = pd.DataFrame(X_train_imputed)

X_train_imputed_df.isna().sum().sum()


np.int64(0)

In [48]:
X_train_imputed_df

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,3.0,29.807687,0.0,0.0,56.4958,male,S,Third,man,True,Southampton,yes,True
1,2.0,29.807687,0.0,0.0,0.0,male,S,Second,man,True,Southampton,no,True
2,1.0,29.807687,0.0,0.0,221.7792,male,S,First,man,True,Southampton,no,True
3,3.0,18.0,0.0,1.0,9.35,female,S,Third,woman,False,Southampton,yes,False
4,2.0,31.0,1.0,1.0,26.25,female,S,Second,woman,False,Southampton,yes,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,3.0,29.807687,0.0,0.0,7.8792,female,Q,Third,woman,False,Queenstown,yes,True
708,1.0,35.0,0.0,0.0,512.3292,female,C,First,woman,False,Cherbourg,yes,True
709,3.0,48.0,1.0,3.0,34.375,female,S,Third,woman,False,Southampton,no,False
710,1.0,47.0,0.0,0.0,38.5,male,S,First,man,True,Southampton,no,True


---
# No redudent code(pipeline tips dataset)

In [49]:
import seaborn as sns
import pandas as pd


In [50]:
df = sns.load_dataset("tips")
df.head()


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [51]:
X = df.drop(columns="total_bill")
y = df["total_bill"]


In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [53]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["category", "object"]).columns

num_cols, cat_cols


(Index(['tip', 'size'], dtype='object'),
 Index(['sex', 'smoker', 'day', 'time'], dtype='object'))

In [54]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [55]:
num_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])


In [56]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ]
)


In [57]:
from sklearn.linear_model import LinearRegression

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])


In [58]:
model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [59]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)


In [60]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

In [61]:
print("Train MAE:", mean_absolute_error(y_train, y_pred_train))
print("Test  MAE:", mean_absolute_error(y_test, y_pred_test))

print("Train RMSE:", root_mean_squared_error(y_train, y_pred_train))
print("Test  RMSE:", root_mean_squared_error(y_test, y_pred_test))

print("Train R²:", r2_score(y_train, y_pred_train))
print("Test  R²:", r2_score(y_test, y_pred_test))


Train MAE: 4.204065642679813
Test  MAE: 4.470042009997702
Train RMSE: 5.822364602019708
Test  RMSE: 5.645674056366515
Train R²: 0.5570216343889893
Test  R²: 0.6240808714290967


In [62]:
print(df['total_bill'].describe())

count    244.000000
mean      19.785943
std        8.902412
min        3.070000
25%       13.347500
50%       17.795000
75%       24.127500
max       50.810000
Name: total_bill, dtype: float64


# Evaluating with MAE

In [63]:
MEAN = 19.785943
MAE = 4.470042009997702
f"{MAE/MEAN * 100:.2f}%"

'22.59%'

In [64]:
Q3 = 24.127500
Q1 = 13.347500 
IQR = Q3 - Q1
f"{MAE/IQR *100:.2f}%" 

'41.47%'

In [65]:
MIN = 3.070000
MAX = 50.810000
RANGE  = MAX - MIN 
f"{MAE / RANGE * 100:.2f}%"

'9.36%'


Using the descriptive statistics of **`total_bill`**, we can evaluate model error relative to the **mean**, the **interquartile range (Q3 − Q1)**, and the **full range (max − min)** to understand performance at different levels of the data distribution.

The **mean total bill** represents a typical bill size and is given by:

$$
\text{Mean} \approx 19.79
$$

With a model **MAE ≈ 4.47**, the normalized error relative to the mean is:

$$
\frac{\text{MAE}}{\text{Mean}} = \frac{4.47}{19.79} \approx 0.226 = 22.6\%
$$

This indicates that the model’s average prediction error is about **23\% of a typical bill**, which is reasonable for a baseline regression model.

---

The **interquartile range (IQR)** measures the spread of the *middle 50\%* of bills and is calculated as:

$$
\text{IQR} = Q3 - Q1 = 24.1275 - 13.3475 = 10.78
$$

Normalizing MAE by the IQR gives:

$$
\frac{\text{MAE}}{\text{IQR}} = \frac{4.47}{10.78} \approx 0.41 = 41\%
$$

This shows that the average prediction error is about **41\% of the typical spread**, indicating **moderate accuracy** within normal cases.

---

The **range** captures the full span of the data, including extreme values, and is computed as:

$$
\text{Range} = \text{Max} - \text{Min} = 50.81 - 3.07 = 47.74
$$

Normalizing MAE by the full range yields:

$$
\frac{\text{MAE}}{\text{Range}} = \frac{4.47}{47.74} \approx 0.094 = 9.4\%
$$

This indicates that the average error is **less than 10\% of the total possible variation**, suggesting the model performs reasonably well when considering all possible bill amounts.

---

**Overall interpretation:**
Relative to the **mean**, the model’s error is moderate; relative to the **IQR**, it shows noticeable uncertainty in typical cases; and relative to the **full range**, it performs well. Together, these perspectives confirm that the model is a **solid baseline**—capturing meaningful signal across the distribution while leaving room for improvement through better feature engineering or more expressive models.


# Evaluating with RMSE

In [66]:
print("Train MAE:", mean_absolute_error(y_train, y_pred_train))
print("Test  MAE:", mean_absolute_error(y_test, y_pred_test))

print("Train RMSE:", root_mean_squared_error(y_train, y_pred_train))
print("Test  RMSE:", root_mean_squared_error(y_test, y_pred_test))

print("Train R²:", r2_score(y_train, y_pred_train))
print("Test  R²:", r2_score(y_test, y_pred_test))

Train MAE: 4.204065642679813
Test  MAE: 4.470042009997702
Train RMSE: 5.822364602019708
Test  RMSE: 5.645674056366515
Train R²: 0.5570216343889893
Test  R²: 0.6240808714290967


In [67]:

print(df['total_bill'].describe())

count    244.000000
mean      19.785943
std        8.902412
min        3.070000
25%       13.347500
50%       17.795000
75%       24.127500
max       50.810000
Name: total_bill, dtype: float64


In [68]:
RMSE = 5.645674056366515
MAE = 4.470042009997702
RMSE/MAE

1.2630024603212662

$$
\frac{\text{RMSE}}{\text{MAE}} = \frac{5.6457}{4.4700} \approx 1.26
$$

This ratio shows that the RMSE is about **26% higher than the MAE**. Since RMSE penalizes larger errors more strongly than MAE, this difference indicates that while most prediction errors are close to the average error captured by MAE, there are some larger errors that increase the RMSE. However, because the ratio is not excessively high, these larger errors are **moderate rather than extreme**. This suggests that the model’s predictions are generally stable, with occasional higher deviations but no severe outliers, which is typical behavior for a baseline linear regression model.


---

RMSE is 1.26 times MAE.

---
Since the RMSE-to-MAE ratio is **1.26**, it means the RMSE is **26% higher than the MAE** ((1.26 - 1 = 0.26)), showing that larger errors exist but are not extreme. This indicates a stable error distribution where occasional higher deviations moderately increase the RMSE.


# Evaluating with r2 score


In [69]:

print("Train MAE:", mean_absolute_error(y_train, y_pred_train))
print("Test  MAE:", mean_absolute_error(y_test, y_pred_test))

print("Train RMSE:", root_mean_squared_error(y_train, y_pred_train))
print("Test  RMSE:", root_mean_squared_error(y_test, y_pred_test))

print("Train R²:", r2_score(y_train, y_pred_train))
print("Test  R²:", r2_score(y_test, y_pred_test))

Train MAE: 4.204065642679813
Test  MAE: 4.470042009997702
Train RMSE: 5.822364602019708
Test  RMSE: 5.645674056366515
Train R²: 0.5570216343889893
Test  R²: 0.6240808714290967


When evaluating a regression model, the **R² score** should be interpreted by first looking at its value on the **test data** and then comparing it with the **training R²**. R² represents the proportion of variance in the target variable that the model explains, so higher values indicate that the model captures more of the underlying signal. However, a high R² is only meaningful if the **training and testing R² values are close to each other**, as a large gap suggests overfitting. An R² close to 1 indicates a near-perfect fit, which is rare in real-world data and may even signal data leakage, while values near 0 indicate little to no predictive power. In practice, an R² in the range of 0.3–0.6 reflects a reasonable baseline model, and values above 0.6 indicate strong explanatory power, provided the test and train scores are similar.


# pending

Model selection (LinearRegression, RandomForest, etc.)


Model evaluation for classification

Hyperparameter tuning

Cross-validation